In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp,to_timestamp,trim,lower,when,max,current_date,col, cast, lit

from delta.tables import delta, DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, Row

In [0]:
%run ../utils/params

In [0]:
%run ../utils/adlsConfig

In [0]:
dfConfig = spark.read.csv(jobConfigPath, inferSchema = True, header = True)
sourceObjectNameList = dfConfig.select("sourceObjectName").rdd.map(lambda x: x[0]).collect()

In [0]:
for source in sourceObjectNameList:
    if source != "dbo.CandidateTaxInfo":
        sourceSystem = "bullhorndtp"
        newSourceObject = source.split(".")[1]
        finalName = sourceSystem + newSourceObject
        query = (f"SELECT * FROM acecurated.{finalName} where dateLastSync > '2023-04-19 06:30:00'")
        df = spark.sql(query)
        if df.isEmpty():
            print("No records for table: ",finalName)
        else:
            deleteQuery = (f"""DELETE FROM acecurated.{finalName} WHERE dateLastSync > '2023-04-19 06:30:00'""")
            spark.sql(deleteQuery)
            print("Rows deleted for table: ", finalName)
            # exec(f"""df{finalName} = spark.sql("{query}")""")
    else:
        pass